# VAST-LoRA: Qwen2.5-3B Slice Matrix on Kaggle T4x2

This notebook clones a pinned VAST-LoRA commit, runs the same 3B async LoRA runner across three regimes, and summarizes whether VAST improves likelihood/calibration in the hard slice.

Regimes:

- `iid_homogeneous`: IID client data, all client LoRA ranks are 8.
- `iid_heterogeneous`: IID client data, client LoRA ranks are 4/8/16.
- `noniid_high_staleness`: label-sharded non-IID data, ranks 4/8/16, high-staleness async trace.

Main comparison: `freshness` vs `vast` vs `mtip`. The primary question is not raw accuracy; it is whether VAST improves sequence NLL / binary NLL / Brier while preserving balanced accuracy in the hard slice.

**Kaggle settings:** enable Internet and use accelerator **GPU T4 x2**. `RUN_MODE = "full"` runs 27 jobs: 3 regimes x 3 methods x 3 seeds. `RUN_MODE = "pilot"` runs only the hard regime with one seed and fewer examples.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/TrgPhan/VASTLoRA.git"
REPO_REF = "0d9d7aa734ed3c9338d17716f887f54d3117e8cb"
RUN_MODE = "full"  # Use "pilot" for a quick smoke test.

WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / "VASTLoRA-slice-run"
RESULT_DIR = WORK_ROOT / "vastlora-3b-slice-results"
assert RUN_MODE in {"pilot", "full"}
print({"repo_ref": REPO_REF, "run_mode": RUN_MODE})

## 1. Clone and install the pinned implementation

In [ ]:
if REPO_DIR.exists():
    assert REPO_DIR.parent == WORK_ROOT
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[scale]"],
    check=True,
)
resolved_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
assert resolved_commit == REPO_REF
print("Installed commit:", resolved_commit)

## 2. Verify T4x2 and cache shared assets

In [ ]:
import torch

subprocess.run(["nvidia-smi"], check=True)
gpu_count = torch.cuda.device_count()
assert gpu_count >= 2, f"Expected Kaggle T4x2, found {gpu_count} CUDA device(s)"
gpu_names = [torch.cuda.get_device_name(index) for index in range(gpu_count)]
print("CUDA devices:", gpu_names)

In [ ]:
from datasets import load_dataset
from huggingface_hub import snapshot_download

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
snapshot_download(MODEL_NAME)
load_dataset("nyu-mll/glue", "sst2")
print("Model and SST-2 are cached for both GPU workers.")

## 3. Build regime configs and dry-run the matrix

In [ ]:
BASE_CONFIG_PATH = REPO_DIR / "configs/kaggle_3b_slice_matrix.json"
RUNNER = REPO_DIR / "scripts/run_kaggle_3b.py"
SUMMARY_SCRIPT = REPO_DIR / "scripts/summarize_kaggle_3b_slices.py"

base_config = json.loads(BASE_CONFIG_PATH.read_text(encoding="utf-8"))
regime_specs = {item["name"]: item for item in base_config["slice_matrix"]["regimes"]}

if RUN_MODE == "pilot":
    regime_names = ["noniid_high_staleness"]
    seeds = [3101]
    methods = ["freshness", "vast", "mtip"]
else:
    regime_names = ["iid_homogeneous", "iid_heterogeneous", "noniid_high_staleness"]
    seeds = base_config["experiment"]["seeds"]
    methods = base_config["experiment"]["methods"]

if RESULT_DIR.exists():
    assert RESULT_DIR.parent == WORK_ROOT
    shutil.rmtree(RESULT_DIR)
RESULT_DIR.mkdir(parents=True)
CONFIG_DIR = RESULT_DIR / "configs"
LOG_DIR = RESULT_DIR / "logs"
CONFIG_DIR.mkdir()
LOG_DIR.mkdir()

def write_regime_config(regime_name):
    spec = regime_specs[regime_name]
    config = json.loads(json.dumps(base_config))
    config["output_dir"] = str(RESULT_DIR)
    config["experiment"]["partition_mode"] = spec["partition_mode"]
    config["experiment"]["client_ranks"] = spec["client_ranks"]
    if RUN_MODE == "pilot":
        config["experiment"]["collected_returns"] = 16
        config["dataset"]["eval_examples"] = 256
    path = CONFIG_DIR / f"{regime_name}.json"
    path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    return path

regime_configs = {name: write_regime_config(name) for name in regime_names}
for regime_name, config_path in regime_configs.items():
    for method in methods:
        subprocess.run(
            [sys.executable, str(RUNNER), "--config", str(config_path), "--method", method, "--dry-run"],
            cwd=REPO_DIR,
            check=True,
        )

## 4. Run the 3B slice matrix

Jobs are launched in pairs so each process sees one T4 through `CUDA_VISIBLE_DEVICES`. Every method in the same regime/seed uses the same partition and async trace.

In [ ]:
jobs = [(regime, method, seed) for regime in regime_names for seed in seeds for method in methods]

def launch_job(regime, method, seed, gpu):
    variant = f"{regime}_{method}"
    log_path = LOG_DIR / f"{variant}_seed{seed}.log"
    log_handle = log_path.open("w", encoding="utf-8")
    env = os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES": str(gpu),
        "PYTHONUNBUFFERED": "1",
        "TOKENIZERS_PARALLELISM": "false",
    })
    command = [
        sys.executable, str(RUNNER),
        "--config", str(regime_configs[regime]),
        "--method", method,
        "--variant", variant,
        "--seed", str(seed),
        "--output-dir", str(RESULT_DIR),
    ]
    process = subprocess.Popen(command, cwd=REPO_DIR, env=env, stdout=log_handle, stderr=subprocess.STDOUT)
    return process, log_handle, log_path

started = time.perf_counter()
for wave_start in range(0, len(jobs), 2):
    wave = jobs[wave_start:wave_start + 2]
    running = [launch_job(regime, method, seed, gpu) for gpu, (regime, method, seed) in enumerate(wave)]
    print("Started:", wave)
    while any(process.poll() is None for process, _, _ in running):
        time.sleep(30)
        active = [item for item, run in zip(wave, running) if run[0].poll() is None]
        print("  still running:", active)
    for (regime, method, seed), (process, handle, log_path) in zip(wave, running):
        handle.close()
        tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]
        print(f"\n--- {regime}/{method} seed={seed}, exit={process.returncode} ---")
        print("\n".join(tail))
        if process.returncode != 0:
            raise RuntimeError(f"{regime}/{method} seed={seed} failed; inspect {log_path}")

wall_minutes = (time.perf_counter() - started) / 60
print(f"All {len(jobs)} jobs completed in {wall_minutes:.1f} minutes.")

## 5. Summarize VAST vs Freshness by regime

In [ ]:
subprocess.run(
    [sys.executable, str(SUMMARY_SCRIPT), "--input-dir", str(RESULT_DIR), "--target-method", "vast"],
    cwd=REPO_DIR,
    check=True,
)

import pandas as pd
from IPython.display import Markdown, display

summary_dir = RESULT_DIR / "slice_summary"
method_summary = pd.read_csv(summary_dir / "method_summary.csv")
paired = pd.read_csv(summary_dir / "paired_comparisons.csv")
regime_summary = pd.read_csv(summary_dir / "regime_summary.csv")
event_slice_summary = pd.read_csv(summary_dir / "event_slice_summary.csv")
verdict = json.loads((summary_dir / "verdict.json").read_text(encoding="utf-8"))

display(Markdown((summary_dir / "verdict.md").read_text(encoding="utf-8")))
display(Markdown("### Regime summary"))
display(regime_summary)
display(Markdown("### Paired comparisons vs Freshness"))
display(paired)
display(Markdown("### Method summary"))
display(method_summary[[
    "regime", "method", "variant", "seed", "partition_mode", "client_ranks",
    "final_accuracy", "final_balanced_accuracy", "final_nll", "final_binary_nll",
    "final_brier", "mean_staleness", "mean_rho_after_warmup", "runtime_seconds",
]])
display(Markdown("### Event slices by tau band and client rank"))
display(event_slice_summary)

## 6. Plot the trade-off

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for method, frame in paired.groupby("method"):
    vast_like = frame.sort_values(["regime", "seed"])
    axes[0].scatter(vast_like["regime"], vast_like["balanced_accuracy_gain_pp"], label=method)
    axes[1].scatter(vast_like["regime"], 100 * vast_like["sequence_nll_relative_change"], label=method)
    axes[2].scatter(vast_like["regime"], 100 * vast_like["binary_nll_relative_change"], label=method)
axes[0].axhline(-0.5, color="black", linewidth=1, linestyle="--")
axes[0].set(title="Balanced accuracy gain vs Freshness", ylabel="pp")
axes[1].axhline(-5.0, color="black", linewidth=1, linestyle="--")
axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].set(title="Sequence NLL relative change", ylabel="%")
axes[2].axhline(5.0, color="black", linewidth=1, linestyle="--")
axes[2].axhline(0.0, color="black", linewidth=1)
axes[2].set(title="Binary NLL relative change", ylabel="%")
for axis in axes:
    axis.tick_params(axis="x", rotation=25)
    axis.grid(alpha=0.25)
    axis.legend()
fig.tight_layout()
plt.show()

## 7. Reproducibility record and archive

In [ ]:
from IPython.display import FileLink

record = {
    "git_commit": resolved_commit,
    "run_mode": RUN_MODE,
    "model": MODEL_NAME,
    "gpus": gpu_names,
    "regimes": regime_names,
    "methods": methods,
    "seeds": seeds,
    "wall_minutes": wall_minutes,
    "verdict": verdict,
}
(RESULT_DIR / "run_record.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
archive = shutil.make_archive(str(WORK_ROOT / "vastlora-3b-slice-results"), "zip", RESULT_DIR)
display(record)
display(FileLink(archive))

### Interpretation rule

- `GO`: in `noniid_high_staleness`, VAST improves sequence NLL by at least 5%, the sequence-NLL CI is below 0, balanced accuracy remains within the non-inferiority margin, and binary NLL/Brier do not regress materially.
- `INCONCLUSIVE`: VAST has some likelihood signal but misses one or more gates.
- `NO_GO`: VAST does not produce a useful likelihood/calibration trade-off against Freshness in the hard 3B slice.

The notebook deliberately does not include fabricated pre-run metrics. Save a Kaggle version after execution to preserve every table, plot, log tail, and verdict.